# Diffusers local test

Dependencies (`torch`, `torchvision`, `diffusers`, `transformers`, `accelerate`, `safetensors`) are managed via `uv` in `pyproject.toml` for this project — no need to `pip install` from inside the notebook. Just make sure the kernel is set to **image-gen (uv)** (top right of the notebook).

This notebook loads [SD-Turbo](https://huggingface.co/stabilityai/sd-turbo), a distilled Stable Diffusion model tuned for 1-4 step inference, and runs it on Apple Silicon via the MPS backend.

In [2]:
import sys
print(sys.executable)

/Users/jaysilvas/dev/ai-build-and-learn/topics/image-gen/.venv/bin/python


In [3]:
import torch
import diffusers

print("torch:", torch.__version__)
print("diffusers:", diffusers.__version__)
print("MPS available:", torch.backends.mps.is_available())

/Users/jaysilvas/dev/ai-build-and-learn/topics/image-gen/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.13.0
diffusers: 0.39.0
MPS available: True


In [4]:
from diffusers import AutoPipelineForText2Image

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device in ("mps", "cuda") else torch.float32

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=dtype)
pipe = pipe.to(device)

print(f"Loaded on device={device}, dtype={dtype}")

[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.
/Users/jaysilvas/dev/ai-build-and-learn/topics/image-gen/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading pipeline components...: 100%|██████████| 5/5 [00:04<00:00,  1.19it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstance

Loaded on device=mps, dtype=torch.float16


In [ ]:
prompt = 'An artfully hand-drawn portrait of a red panda puffing smoke rings from a tobacco pipe, Natural History Illustration, Botanical-styled Illustration. Negative prompt: (canvas frame, watermark, signature, username, artist name:1.1)'

# SD-Turbo is distilled for 1-4 step inference with guidance_scale=0.0
num_inference_steps = 2
guidance_scale = 0.0
image = pipe(prompt=prompt, num_inference_steps=num_inference_steps, guidance_scale=guidance_scale).images[0]
image

In [ ]:
from gallery import store

store.save_image(
    image,
    prompt=prompt,
    model="stabilityai/sd-turbo",
    params={
        "num_inference_steps": num_inference_steps,
        "guidance_scale": guidance_scale,
        "device": device,
        "dtype": str(dtype),
    },
)

## Video generation

Now testing text-to-video with [damo-vilab/text-to-video-ms-1.7b](https://huggingface.co/damo-vilab/text-to-video-ms-1.7b), a lightweight (~1.7B param) diffusion model that generates short video clips directly from a text prompt — a reasonable fit for local testing on Apple Silicon, similar in spirit to SD-Turbo above.

Requires `imageio[ffmpeg]` (added to `pyproject.toml`) so `diffusers.utils.export_to_video` can write the frames out as an mp4.

In [6]:
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler

video_pipe = DiffusionPipeline.from_pretrained("damo-vilab/text-to-video-ms-1.7b", torch_dtype=dtype, variant="fp16" if dtype == torch.float16 else None)
video_pipe.scheduler = DPMSolverMultistepScheduler.from_config(video_pipe.scheduler.config)
video_pipe = video_pipe.to(device)

print(f"Loaded video pipeline on device={device}, dtype={dtype}")

Loading pipeline components...: 100%|██████████| 5/5 [00:00<00:00, 12.06it/s]
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


Loaded video pipeline on device=mps, dtype=torch.float16


In [68]:
from diffusers.utils import export_to_video

video_prompt = "following shot of a great dane running through tall grass in a mountain meadow at sunrise, alpenglow, cinematic lighting, 8k, ultra realistic, hyper detailed, photorealistic, trending on artstation"

num_inference_steps = 25
num_frames = 40
fps = 10
frames = video_pipe(prompt=video_prompt, num_inference_steps=num_inference_steps, num_frames=num_frames).frames[0]
video_path = export_to_video(frames, fps=fps)
print("Saved video to:", video_path)

100%|██████████| 25/25 [05:00<00:00, 12.02s/it]


Saved video to: /private/tmp/tmpy4_2cxpy.mp4


In [66]:
from IPython.display import Video

Video(video_path, embed=True)

In [ ]:
from gallery import store

store.save_video(
    video_path,
    prompt=video_prompt,
    model="damo-vilab/text-to-video-ms-1.7b",
    params={
        "num_inference_steps": num_inference_steps,
        "num_frames": num_frames,
        "fps": fps,
        "device": device,
        "dtype": str(dtype),
    },
)

## Gallery

Every `store.save_image(...)` / `store.save_video(...)` call above appends the media file plus its prompt, model, and generation parameters to `gallery/metadata.json` (media saved under `gallery/media/`).

To browse everything generated so far in a masonry-style gallery (with per-item delete), run in a separate terminal:

```bash
uv run python gallery/app.py
```

Then open http://127.0.0.1:5050.